# 04 - Training GNN Models

## Phát hiện Giao dịch Gian lận dựa trên Mạng Neural Đồ thị

Notebook này train và so sánh 3 kiến trúc GNN:
1. **HeteroGraphSAGE** - Mean aggregation với SAGEConv
2. **HeteroGAT** - Multi-head attention với GATConv
3. **HeteroRGCN** - Relation-specific weights với GraphConv

Tất cả sử dụng:
- Focal Loss cho class imbalance
- NeighborLoader cho mini-batch training
- CosineAnnealing LR scheduling
- Early stopping
- Gradient clipping

In [1]:
import sys
sys.path.insert(0, '..')

from src.config import set_seed, DEVICE, HIDDEN_DIM, NUM_LAYERS, NUM_HEADS, DROPOUT
from src.data_loader import IEEECISDataLoader
from src.graph_builder import HeteroGraphBuilder
from src.sampling import ImbalanceSampler
from src.models import HeteroGraphSAGE, HeteroGAT, HeteroRGCN
from src.trainer import GNNTrainer
from src.evaluator import ModelEvaluator

set_seed(42)
print(f"Device: {DEVICE}")

Device: cpu


## 1. Chuẩn bị dữ liệu

In [2]:
# Load data
loader = IEEECISDataLoader()
df = loader.load()

# Build graph
builder = HeteroGraphBuilder(df)
data = builder.build()

# Create samplers and loaders
sampler = ImbalanceSampler(data)
loaders = sampler.get_all_loaders()
loss_fn = sampler.get_focal_loss()

# Model parameters
metadata = data.metadata()
in_channels = data['txn'].x.shape[1]
print(f"\nInput channels: {in_channels}")
print(f"Hidden dim: {HIDDEN_DIM}")
print(f"Num layers: {NUM_LAYERS}")


LOADING DATA
Loaded 590,540 transactions
Fraud rate: 3.50% (20,663 fraud transactions)
txn_index range: 0 to 590539
Columns: ['txn_index', 'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'DeviceInfo', 'DeviceType', 'id_30', 'id_31']
[load] completed in 1.12s

BUILDING HETEROGENEOUS GRAPH
Transaction nodes: 590,540
Building transaction features...
  Features shape: (590540, 6)
  Labels shape: (590540,) (fraud: 20,663)
[build_features] completed in 0.08s

BUILDING ENTITY INDEXERS
  card1: 8,419 unique values
  card2: 500 unique values
  card3: 90 unique values
  card4: 4 unique values
  card5: 91 unique values
  card6: 4 unique values
  addr1: 171 unique values
  addr2: 47 unique values
  p_email: 59 unique values
  r_email: 60 unique values
  device: 1,149 unique values
  devtype: 2 unique values
  os: 74 unique values
  browser: 111 unique values

Adding entity node

## 2. Train HeteroGraphSAGE

In [3]:
set_seed(42)

model_sage = HeteroGraphSAGE(
    metadata=metadata,
    in_channels=in_channels,
    hidden_channels=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
)

trainer_sage = GNNTrainer(model_sage, loss_fn, model_name="HeteroSAGE")
history_sage = trainer_sage.train(loaders['train'], loaders['val'])


TRAINING HETEROSAGE
Device: cpu
Epochs: 50, Patience: 10
Optimizer: Adam (lr=1.0e-03)
Epoch   1/50 | Train Loss: 0.0235 | Val Loss: 0.0260 | Val F1: 0.0002 | Val AUC: 0.6055 | Time: 1211.7s
Model saved to D:\KLTN\Code\gian_lan_ieee\models\HeteroSAGE_best.pt


KeyboardInterrupt: 

## 3. Train HeteroGAT

In [ ]:
set_seed(42)

model_gat = HeteroGAT(
    metadata=metadata,
    in_channels=in_channels,
    hidden_channels=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
)

trainer_gat = GNNTrainer(model_gat, loss_fn, model_name="HeteroGAT")
history_gat = trainer_gat.train(loaders['train'], loaders['val'])

## 4. Train HeteroRGCN

In [ ]:
set_seed(42)

model_rgcn = HeteroRGCN(
    metadata=metadata,
    in_channels=in_channels,
    hidden_channels=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
)

trainer_rgcn = GNNTrainer(model_rgcn, loss_fn, model_name="HeteroRGCN")
history_rgcn = trainer_rgcn.train(loaders['train'], loaders['val'])

## 5. So sánh Training Curves

In [ ]:
evaluator = ModelEvaluator()

# Plot training curves for each model
evaluator.plot_training_curves(history_sage, "HeteroSAGE")
evaluator.plot_training_curves(history_gat, "HeteroGAT")
evaluator.plot_training_curves(history_rgcn, "HeteroRGCN")

In [ ]:
from IPython.display import Image, display
from pathlib import Path

metrics_dir = Path('../output/metrics')
for img_path in sorted(metrics_dir.glob('training_curves_*.png')):
    print(f"\n--- {img_path.stem} ---")
    display(Image(filename=str(img_path), width=700))